# 1 ヒットのスペクトルとゲインマップ

> **準備中:** ゲインマップは **Ba の線源のデータ**で作る必要があります。そのデータの準備ができるまで、
> このノートブックは Per02 で **1 ヒットのスペクトルを作るところ(6 節)まで**を扱います。7 節(ピークのフィットとゲインマップ)は
> Ba のデータで行う予定の手順の骨組みで、まだ使えません。

TPC_Calibration は、FADC で測った電荷 Q [pC] をエネルギー E [keV] に直します。
直し方は「アノードの FADC チャンネル(ASIC)× カソードの FADC チャンネル」の組ごとの 1 次式です。

$$ E = a\,Q + b $$

この a, b の表が**ゲインマップ**(設定ファイルの `TPC_Calibration.ParameterFile`、Per02 では `param/20190528/Calibration20190130.dat`)です。
Per02 は 1 パネルで、アノード・カソードとも 3 ボード × 4 チャンネル = 12 チャンネルなので、表は 12 × 12 = 144 行です。

```
0   0   0.0212138 0.059728     <- アノード ch, カソード ch, a, b
1   0   0.0179809 -0.00561513
...
```

(TPC_Calibration は読み込んだ a, b に `Cal_Modify` を掛けて使います。)

このノートブックでは、

1. 生データツリーの波形から、**閾値を選んで** CalcFADC と同じ電荷を計算する
2. 「1 ヒット」(アノード側・カソード側とも反応した領域が 1 つだけ)のイベントを選び、領域の組ごとの電荷のスペクトルを作る。
   **領域は ASIC(FADC チャンネル)ごとか、ボードごとかを選べる**
3. スペクトルの既知のピークをガウス関数でフィットし、ピークの電荷とエネルギーの関係(複数のピーク)から a, b を求めて、ゲインマップの形式で書き出す

という手順で進めます。1 ヒットのスペクトルの作り方は、EricaNext の開発版(`Erica_git_dev`)の CalcFADC の
`Anode_1Hit_A**_C**`(ASIC ごと)、`Cathode_1Hit_board_A*_C*`(ボードごと)と同じです。

In [ ]:
import sys, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CACHE = REPO / "tutorials_caldata" / "cache"   # チェーンの出力の置き場(チュートリアル間で共有)
CACHE.mkdir(exist_ok=True)

## 1. データの準備

データ(20190701 の較正データ **Per02**)の入手方法と置き場所は、
**[1_DataIO のノートブック](../1_DataIO/DataIO_example.ipynb) の「1. データのダウンロード」**を見てください。
1_DataIO の手順でデータを置いていれば、ここでは何もする必要はありません。

下のセルで、設定ファイル `config/smile2p_20190701_Per02.json` が指すデータが見つかるかだけ確認します
(`MISSING` と出たら 1_DataIO の 1 節の手順でデータをコピーしてください)。

In [ ]:
CONFIG = "smile2p_20190701_Per02"
cfg = ec.load(CONFIG)
print("data_top:", ec.data_top())
for path in [cfg["parameters"]["ReadData"]["data_dir"], cfg["parameters"]["HA_Calc"]["parameter_dir"]]:
    print("OK     " if Path(path).is_dir() else "MISSING", path)

## 2. 較正用の生データツリーを作る

較正には**全イベント**の生データツリー(`ReadData → RawTreeWriter`、中身は `../1_DataIO/modules/8_RawTreeWriter/`)を使います。
ドリフト速度とゲインマップのノートブックで同じファイルを使うので、どちらか一方で一度作れば十分です。

ターミナル(コマンドプロンプト)では、リポジトリの `run_script` ディレクトリで次を実行するのと同じです。

```bash
cd EricaNext_try/run_script
python3 run_chain.py smile2p_20190701_Per02 -o Per02_raw_cal_thr10.root --raw --compression 505 \
    --set RawTreeWriter.save_threshold=10.0 --set RawTreeWriter.save_ha=false
```

| 引数 | 意味 |
|---|---|
| `smile2p_20190701_Per02` | 設定ファイル `config/smile2p_20190701_Per02.json` |
| `-o Per02_raw_cal_thr10.root` | 出力ファイル |
| (`-n` なし) | 全イベント(Per02 は 815,657 イベント) |
| `--raw` | `ReadData → RawTreeWriter` だけを実行する |
| `--compression 505` | zstd で圧縮する |
| `--set RawTreeWriter.save_threshold=10.0` | 波形の最大値がベースライン + 10 ADU 以下のチャンネルは波形を保存しない(ファイルを小さくするため)。**電荷の閾値は 10 以上なら後から自由に変えられる**(閾値 10 以上なら、保存しなかったチャンネルの電荷はどのみち「閾値を超えるサンプルなし」になる)。小数で書く(`10` だと整数になり、型が合わずに止まる) |
| `--set RawTreeWriter.save_ha=false` | HA の ADC は保存しない(TPC の較正には使わない) |

`--set モジュール名.パラメータ名=値` は設定ファイルの値をその場で上書きするオプションです(何回でも書けます)。

**時間と大きさの目安:** 約 80 分、約 4.6 GB(データを手元のディスクに置いた場合)。
下のセルは同じコマンドを Python から実行します。出力が既にあれば再実行しません。

In [ ]:
RAW = CACHE / "Per02_raw_cal_thr10.root"
cmd = [sys.executable, str(RUN_CHAIN), CONFIG, "-o", str(RAW), "--raw", "--compression", "505",
       "--set", "RawTreeWriter.save_threshold=10.0", "--set", "RawTreeWriter.save_ha=false"]
print(" ".join(cmd))
if not RAW.exists():
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()
raw = ROOT.RDataFrame("RawTreeWriter/raw", str(RAW))
print(raw.Count().GetValue(), "events")

## 3. 設定

| 変数 | 意味 |
|---|---|
| `THRESHOLD` | 電荷を積分するサンプルの閾値 [ADU](ベースラインからの高さ)。CalcFADC の既定は 20。生データツリーを `save_threshold=10` で作ったので **10 以上**にする |
| `REGION` | `"asic"`: FADC チャンネル(ASIC)ごと、`"board"`: ボード(4 チャンネル)ごと |

In [ ]:
THRESHOLD = 20.0
REGION = "board"          # "asic" または "board"
N_BOARD, N_CH = 3, 4      # Per02: アノード・カソードとも 3 ボード × 4 チャンネル
assert THRESHOLD >= 10.0, "生データツリーは save_threshold=10 で作ったので、閾値は 10 以上にする"

## 4. 電荷を計算する(CalcFADC と同じ)

下の C++ の関数 `charges` は CalcFADC と同じ計算です(`../1_DataIO/modules/2_CalcFADC/` で、チェーン内の値と一致することを確認した Python の関数を
C++ に書き直したもの。全イベントを速く処理するため)。

- ベースライン: 最初の 30 サンプルの平均
- 電荷: サンプル 30〜499 のうち、ベースラインから `THRESHOLD` を超えたサンプルの (値 − ベースライン) の和 × 2 mV × 20 ns / 50 Ω → pC
- 閾値を超えるサンプルが無いチャンネルは −100 × 0.8 = −80(CalcFADC と同じ)。波形を保存していないチャンネルもこれになる

チャンネル番号は生データツリーの `fadc_ch` と同じで、0–11 がアノード、12–23 がカソード(ボード × 4 + チャンネル)です。

※ 開発版(`Erica_git_dev`)の CalcFADC は単位が ADU × クロック(0.8 を掛けない)で、積分がサンプル 511 まで、閾値を超えないチャンネルは 0 です。
そのためスペクトルの横軸の値は開発版の図と 0.8 倍ほど違います。ここではゲインマップを掛ける相手である CalcFADC(本流)の定義にそろえています。

In [ ]:
ROOT.gInterpreter.Declare(r"""
// CalcFADC と同じ電荷 [pC]。24 チャンネル(0-11 アノード, 12-23 カソード)
ROOT::RVecD charges(const ROOT::RVec<short>& ch, const ROOT::RVec<short>& wave, double threshold) {
  const int N = 512;
  ROOT::RVecD q(24, -100. * 2. * 20. / 50.);   // 閾値を超えるサンプルなし(波形を保存していないチャンネルも)
  for (size_t i = 0; i < ch.size(); i++) {
    const short* w = &wave[i * N];
    double mean = 0;
    for (int k = 0; k < 30; k++) mean += (double)w[k];
    mean /= 30;
    double charge = 0;
    int n_over = 0;
    for (int k = 30; k < 500; k++)
      if ((double)w[k] - mean > threshold) { charge += (double)w[k] - mean; n_over++; }
    if (n_over == 0) charge = -100;
    q[ch[i]] = charge * 2. * 20. / 50.;
  }
  return q;
}
""")
df = raw.Define("q", f"charges(fadc_ch, fadc_wave, {THRESHOLD})")

## 5. 何チャンネルが反応するか

イベントごとに、電荷が正(閾値を超えたサンプルがある)のチャンネルの数を数えます。
Per02 では、反応したボードの中の 4 チャンネルがほとんど同時に正になるので、**閾値 20 では ASIC ごとの 1 ヒットはほとんど無く**、
ボードごとの 1 ヒットを使うことになります。閾値を上げると ASIC ごとの 1 ヒットが増えます(下の表で確認できます)。

In [ ]:
n_pos = {"anode": df.Define("n", "Sum(Take(q, 12) > 0)").Histo1D(("na", "", 13, -0.5, 12.5), "n"),
         "cathode": df.Define("n", "Sum(Take(q, -12) > 0)").Histo1D(("nc", "", 13, -0.5, 12.5), "n")}
fig, ax = plt.subplots(figsize=(7, 4))
for name, h in n_pos.items():
    y = [h.GetBinContent(i) for i in range(1, 14)]
    ax.step(range(13), y, where="mid", label=name)
ax.set_xlabel("number of channels with Q > 0 in an event"); ax.set_ylabel("events"); ax.set_yscale("log"); ax.legend()
ax.set_title(f"threshold = {THRESHOLD} ADU")
fig.tight_layout()

## 6. 1 ヒットの選択とスペクトル

開発版の CalcFADC と同じ選び方です。

| `REGION` | 1 ヒットの条件 | 詰める電荷 | 領域の数 |
|---|---|---|---|
| `"asic"` | 電荷が正のチャンネルが、アノード 12 本中 1 本、カソード 12 本中 1 本だけ | そのチャンネルの電荷 | 12 × 12 |
| `"board"` | 電荷が正のチャンネルを持つボードが、アノード 3 枚中 1 枚、カソード 3 枚中 1 枚だけ | そのボードの、電荷が正のチャンネルの和 | 3 × 3 |

下の関数 `one_hit` は {アノードの領域, カソードの領域, アノードの電荷, カソードの電荷} を返します(1 ヒットでなければ領域が −1)。
領域の組を 1 つの番号(アノードの領域 × カソードの領域の数 + カソードの領域)にして、
「領域の組 × 電荷」の 2 次元ヒストグラムに詰め、あとで組ごとに取り出します。

In [ ]:
ROOT.gInterpreter.Declare(r"""
// 1 ヒットの選択(EricaNext 開発版 CalcFADC と同じ)。{アノードの領域, カソードの領域, Qa, Qc}、1 ヒットでなければ領域 -1
ROOT::RVecD one_hit(const ROOT::RVecD& q, bool board) {
  int na = 0, nc = 0, ia = -1, ic = -1;
  double qa = 0, qc = 0;
  if (!board) {
    for (int i = 0; i < 12; i++) if (q[i] > 0) { na++; ia = i; }
    for (int i = 0; i < 12; i++) if (q[12 + i] > 0) { nc++; ic = i; }
    if (na == 1 && nc == 1) { qa = q[ia]; qc = q[12 + ic]; }
  } else {
    for (int b = 0; b < 3; b++) {
      double sa = 0, sc = 0; bool fa = false, fc = false;
      for (int k = 0; k < 4; k++) {
        if (q[4 * b + k] > 0) { fa = true; sa += q[4 * b + k]; }
        if (q[12 + 4 * b + k] > 0) { fc = true; sc += q[12 + 4 * b + k]; }
      }
      if (fa) { na++; ia = b; qa = sa; }
      if (fc) { nc++; ic = b; qc = sc; }
    }
  }
  if (na != 1 || nc != 1) return {-1., -1., 0., 0.};
  return {double(ia), double(ic), qa, qc};
}
""")
N_REG = N_BOARD if REGION == "board" else N_BOARD * N_CH          # 片側の領域の数
Q_BINS = (2000, 0., 8000.)                                        # 電荷 [pC] のビン
hit = (df.Define("h", f"one_hit(q, {'true' if REGION == 'board' else 'false'})")
         .Filter("h[0] >= 0")
         .Define("pair", f"h[0] * {N_REG} + h[1]")
         .Define("qa", "h[2]").Define("qc", "h[3]"))
H2 = {"Anode": hit.Histo2D(("spec_a", "", N_REG * N_REG, -0.5, N_REG * N_REG - 0.5, *Q_BINS), "pair", "qa"),
      "Cathode": hit.Histo2D(("spec_c", "", N_REG * N_REG, -0.5, N_REG * N_REG - 0.5, *Q_BINS), "pair", "qc")}
H2 = {k: v.GetValue() for k, v in H2.items()}   # ここで全イベントを読む(数分)

def spectrum(side, ia, ic):
    """領域の組 (ia, ic) の 1 ヒットのスペクトル(TH1D)。名前は開発版と同じ形"""
    b = ia * N_REG + ic + 1
    tag = "board_" if REGION == "board" else ""
    return H2[side].ProjectionY(f"{side}_1Hit_{tag}A{ia}_C{ic}", b, b)

counts = np.array([[spectrum("Anode", i, j).GetEntries() for j in range(N_REG)] for i in range(N_REG)])
print(f"REGION = {REGION}, THRESHOLD = {THRESHOLD}: 1 ヒットのイベント {int(counts.sum())}")

In [ ]:
fig, ax = plt.subplots(figsize=(5 + 0.2 * N_REG, 4 + 0.2 * N_REG))
m = ax.imshow(counts.T, origin="lower", cmap="viridis")
fig.colorbar(m, ax=ax, label="1-hit events")
ax.set_xlabel(f"anode {REGION}"); ax.set_ylabel(f"cathode {REGION}"); ax.set_title(f"1-hit events (threshold {THRESHOLD} ADU)")
fig.tight_layout()

領域の組ごとのスペクトルです(`REGION = "asic"` のときは 144 組あるので、イベントの多い組から 9 組を描きます)。

In [ ]:
order = np.dstack(np.unravel_index(np.argsort(counts, axis=None)[::-1], counts.shape))[0][:9]
pairs = [(i, j) for i in range(N_REG) for j in range(N_REG)] if REGION == "board" else [tuple(p) for p in order]
nrow = int(np.ceil(len(pairs) / 3))
fig, ax = plt.subplots(nrow, 3, figsize=(15, 3.6 * nrow), squeeze=False)
for a, (i, j) in zip(ax.flat, pairs):
    for side in ("Anode", "Cathode"):
        h = spectrum(side, i, j)
        x = [h.GetBinCenter(k) for k in range(1, h.GetNbinsX() + 1)]
        y = [h.GetBinContent(k) for k in range(1, h.GetNbinsX() + 1)]
        a.step(x, y, where="mid", label=side)
    a.set_xlim(0, 4000); a.set_xlabel("charge [pC]"); a.set_ylabel("events / 4 pC")
    a.set_title(f"1Hit A{i} C{j}"); a.legend()
for a in ax.flat[len(pairs):]:
    a.axis("off")
fig.tight_layout()

## 7. 既知のピークのフィットとゲインマップ

スペクトルの中の、エネルギーが分かっているピーク(線源の線、特性 X 線など)をガウス関数でフィットし、
ピークの電荷 Q とエネルギー E の点に直線 E = a Q + b を当てます。ゲインマップは**アノードの電荷**に掛けるので(TPC_Calibration)、
アノードのスペクトルを使います。

`PEAKS` に、使うピークのエネルギー [keV] と、そのピークを探す電荷の範囲 [pC] を書きます(2 本以上で a と b、1 本だけなら b = 0 として a だけ)。

**準備中:** ゲインマップは Ba の線源のデータで作ります(Per02 の Cs-137 のデータでは作れません)。
Ba のデータの生データツリーを 2 節と同じ手順で作り、`RAW` をそのファイルにして、Ba のスペクトルのピークを `PEAKS` に書く予定です。
それまでは `PEAKS` は空で、下のセルは何もしません(書き出すファイルは元の表と同じ内容になります)。

In [ ]:
PEAKS = [
    # {"energy": 5.9,  "range": (100., 300.)},     # 例: ピークのエネルギー [keV] と、探す電荷の範囲 [pC]
    # {"energy": 22.1, "range": (500., 900.)},
]

def fit_peak(h, lo, hi):
    """範囲 [lo, hi] の最大のビンを中心に、±(範囲の 1/4) でガウス関数をフィット -> (平均, 誤差, σ)"""
    h.GetXaxis().SetRangeUser(lo, hi)
    c = h.GetBinCenter(h.GetMaximumBin())
    h.GetXaxis().UnZoom()
    w = (hi - lo) / 4.
    f = ROOT.TF1("peak", "gaus", c - w, c + w)
    f.SetParameters(h.GetMaximum(), c, w / 2)
    h.Fit(f, "QRN")
    return f.GetParameter(1), f.GetParError(1), abs(f.GetParameter(2))

gain = {}   # (ia, ic) -> (a, b)
if not PEAKS:
    print("PEAKS が空です。使うピークを書いてから実行してください。")
else:
    E = np.array([p["energy"] for p in PEAKS])
    for i in range(N_REG):
        for j in range(N_REG):
            h = spectrum("Anode", i, j)
            if h.GetEntries() < 100:
                continue
            Q = np.array([fit_peak(h, *p["range"])[0] for p in PEAKS])
            if len(PEAKS) == 1:
                gain[(i, j)] = (E[0] / Q[0], 0.)
            else:
                a, b = np.polyfit(Q, E, 1)
                gain[(i, j)] = (a, b)
    print(len(gain), "組の a, b を求めた")

### ゲインマップの書き出し

TPC_Calibration が読む形(アノード ch, カソード ch, a, b の 144 行)で書きます。
`REGION = "board"` のときは、ボードの組の a, b を、その中の 4 × 4 = 16 個の ASIC の組すべてに使います。
1 ヒットのイベントが少なくて求められなかった組は、データに付いている表の値をそのまま使います。

In [ ]:
REF_CAL = Path(cfg["parameters"]["TPC_Calibration"]["ParameterFile"])
ref = {}
for line in open(REF_CAL):
    na, nc, a, b = line.split()
    ref[(int(na), int(nc))] = (float(a), float(b))

new = {}
for (na, nc), ab in ref.items():
    key = (na // N_CH, nc // N_CH) if REGION == "board" else (na, nc)
    new[(na, nc)] = gain.get(key, ab)

OUT_CAL = CACHE / f"Calibration_Per02_{REGION}_thr{THRESHOLD:g}.dat"
with open(OUT_CAL, "w") as fout:
    for (na, nc), (a, b) in sorted(new.items(), key=lambda kv: (kv[0][1], kv[0][0])):
        fout.write(f"{na}\t{nc}\t{a:g} {b:g}\n")
print("wrote", OUT_CAL, "(このノートブックで求めた組:", sum(1 for k in ref if ((k[0] // N_CH, k[1] // N_CH) if REGION == "board" else k) in gain), "/ 144)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ra = np.array([[ref[(i, j)][0] for j in range(12)] for i in range(12)])
na_ = np.array([[new[(i, j)][0] for j in range(12)] for i in range(12)])
for a, z, t in zip(ax, [ra, na_], [REF_CAL.name, OUT_CAL.name]):
    m = a.imshow(z.T, origin="lower", cmap="viridis")
    fig.colorbar(m, ax=a, label="a [keV/pC]")
    a.set_xlabel("anode ch"); a.set_ylabel("cathode ch"); a.set_title(t)
fig.tight_layout()

作ったゲインマップを解析に使うときは、`run_chain.py` に `--set TPC_Calibration.ParameterFile=<パス>` を付けるか、
設定ファイルの `TPC_Calibration.ParameterFile` を書き換えます(`Cal_Modify` も合わせて見直してください)。